# 12 — Features e Pipelines

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 12/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Objetivos de aprendizagem
Descrever pipelines sem vazamento e testar transformações.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
StringIndexer, OneHotEncoder, VectorAssembler e separação de treino.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, average_precision_score, roc_auc_score

v = pd.read_csv('datasets/flights.csv').dropna(subset=['carrier','origin','distance','arr_delay','year','month','day']).copy()
v['distance'] = pd.to_numeric(v['distance'],errors='coerce')
v = v.dropna(subset=['distance'])
v['data_voo'] = pd.to_datetime(v[['year','month','day']],errors='coerce')
v = v.dropna(subset=['data_voo'])
v['atrasado'] = (v['arr_delay'] > 15).astype(int)
# Separação por DATA, sem compartilhar dias entre os conjuntos.
dias = sorted(v['data_voo'].unique())
assert len(dias) >= 10, 'Amostra temporal insuficiente'
corte = dias[int(len(dias)*0.75)]
treino = v.loc[v['data_voo']<corte].copy()
teste = v.loc[v['data_voo']>=corte].copy()
assert not treino.empty and not teste.empty
assert treino['data_voo'].max() < teste['data_voo'].min()
assert treino['atrasado'].nunique()==2 and teste['atrasado'].nunique()==2
atributos = ['carrier','origin','distance']
Xtr, ytr = treino[atributos],treino['atrasado']
Xte, yte = teste[atributos],teste['atrasado']
# Encoder e scaler ajustados EXCLUSIVAMENTE dentro do pipeline, sobre o treino.
prep = ColumnTransformer([
    ('cat',OneHotEncoder(handle_unknown='ignore'),['carrier','origin']),
    ('dist',StandardScaler(),['distance'])
])
model = Pipeline([('prep',prep),('clf',LogisticRegression(solver='liblinear',max_iter=500,random_state=42))])
model.fit(Xtr,ytr)
assert model.named_steps['clf'].n_iter_[0] < 500, 'Solver não convergiu'
pred = model.predict(Xte)
proba = model.predict_proba(Xte)[:,1]
print('Datas: treino até',treino['data_voo'].max().date(), '| teste desde',teste['data_voo'].min().date())
print('Amostras treino/teste:',len(Xtr),len(Xte),'| prevalência teste:',round(yte.mean(),3))

print('Balanced accuracy:',round(balanced_accuracy_score(yte,pred),3))
print('Average precision:',round(average_precision_score(yte,proba),3))
print('AP da baseline aleatória: prevalência da classe positiva no TESTE =',round(yte.mean(),3))
print('Alvo é derivado de arr_delay, que NÃO entra nas features. Teste temporal é demonstrativo.')


Datas: treino até 2014-09-30 | teste desde 2014-10-01
Amostras treino/teste: 7538 2387 | prevalência teste: 0.159
Balanced accuracy: 0.5
Average precision: 0.175
AP da baseline aleatória: prevalência da classe positiva no TESTE = 0.159
Alvo é derivado de arr_delay, que NÃO entra nas features. Teste temporal é demonstrativo.


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
print('Baseline classe majoritária no teste:',round(max(yte.mean(),1-yte.mean()),3))
print('O corte temporal mitiga mistura aleatória de épocas, mas não garante validação externa.')
print('Atributos aprendidos somente no treino; não foram usadas variáveis pós-voo nas features.')


Baseline classe majoritária no teste: 0.841
O corte temporal mitiga mistura aleatória de épocas, mas não garante validação externa.
Atributos aprendidos somente no treino; não foram usadas variáveis pós-voo nas features.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 12 reuniu conceitos e verificações de **transformações no treino e avaliação temporal**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** pipeline treinado somente no treino. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **pipeline treinado somente no treino** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **pipeline treinado somente no treino** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Laboratório opcional — Spark local real

Requer `RUN_SPARK_LABS=1` e `SPARK_LAB_PYTHON` apontando para um ambiente isolado com PySpark e Java. O teste falha explicitamente se executado incorretamente; resultados locais não comprovam cluster distribuído.


In [4]:
# Laboratório Spark real opcional, executado em interpretador isolado.
# Ative SOMENTE quando o ambiente PySpark e o Java já estiverem preparados.
import os, sys, subprocess, textwrap
if os.environ.get('RUN_SPARK_LABS') == '1':
    interpretador = os.environ.get('SPARK_LAB_PYTHON', sys.executable)
    assert os.path.isfile(interpretador), f'Interpretador não encontrado: {interpretador}'
    programa = textwrap.dedent(r'''from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler,StandardScaler
s=SparkSession.builder.master('local[2]').appName('ep-area10-12').config('spark.ui.enabled','false').getOrCreate()
try:
    s.sparkContext.setLogLevel('ERROR')
    df=s.createDataFrame([(1.0,120.0),(2.0,240.0),(3.0,360.0),(4.0,480.0)],['x','distancia'])
    assembler=VectorAssembler(inputCols=['x','distancia'],outputCol='features')
    base=assembler.transform(df)
    scaler=StandardScaler(inputCol='features',outputCol='scaled',withMean=False,withStd=True).fit(base)
    feats=scaler.transform(base).select('scaled').collect()
    assert len(feats)==4 and feats[0].scaled.size==2
    print('Pipeline de features Spark ML:',len(feats),'linhas (exemplo sintético)')
    print('SPARK_LAB_OK_12')
finally:s.stop()''')
    ambiente = os.environ.copy()
    ambiente.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
    ambiente['PYSPARK_PYTHON'] = interpretador
    ambiente['PYSPARK_DRIVER_PYTHON'] = interpretador
    resultado = subprocess.run([interpretador, '-c', programa], cwd=os.getcwd(),
                               env=ambiente, text=True, capture_output=True, timeout=150)
    if resultado.stdout: print(resultado.stdout.strip())
    if resultado.returncode:
        print(resultado.stderr[-3500:])
        raise RuntimeError('O LABORATÓRIO SPARK 12 FALHOU; não registrar como aprovado.')
    assert 'SPARK_LAB_OK_12' in resultado.stdout, 'Marcador de sucesso Spark não encontrado'
else:
    print('SPARK NÃO EXECUTADO (opcional). Ative RUN_SPARK_LABS=1 e SPARK_LAB_PYTHON.')


Pipeline de features Spark ML: 4 linhas (exemplo sintético)
SPARK_LAB_OK_12


## Exportação de resultados

Uma única exportação consolidada das métricas do teste temporal.


In [5]:
from pathlib import Path
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
metricas = pd.DataFrame([{'modelo':'logistic_regression','treino':len(Xtr),'teste':len(Xte),
    'prevalencia_teste':float(yte.mean()),
    'balanced_accuracy':float(balanced_accuracy_score(yte,pred)),
    'average_precision':float(average_precision_score(yte,proba))}])
metricas.to_csv(OUT/'12_avaliacao_temporal.csv',index=False)
print('Exportado: outputs/12_avaliacao_temporal.csv')


Exportado: outputs/12_avaliacao_temporal.csv
